# B3MX — Mask-source mixing and count ensembling (CPU)

## Why
Every attempt to choose locally between the sparse and dense regimes (verification gates, rule A, selective negatives, local zoom) failed. Two remaining ideas avoid a regime decision:
1. **Mask-source mixing.** In Z-LM0, the zoomed **automatic** masks behave well in sparse scenes (B0-Z[LM0]: S1 6.45, passes T1 and T2′) while the zoomed **guided** masks appear to cause the sparse surplus (GPV_R4-Z[LM0]: S1 9.40). Pooling zoomed automatic masks with **unzoomed** guided masks may keep zoom's dense recall without the guided part-masks.
2. **Count ensembling.** LM0-based systems undercount dense scenes; zoom-based systems over-count sparse ones. Averaging their counts partly cancels opposite-signed errors, with no gate.

## Pre-declared design (before any B3MX result)
| Item | Rule |
|---|---|
| **MixA** | B0 masks from Z-LM0 (zoomed, Large, masked) + guided masks from LM0 (unzoomed, Large, masked) → GPV_R4 (NMS 0.5 + pool) |
| **MixB** (diagnostic) | B0 masks from LM0 + guided masks from Z-LM0 |
| Counting on each mix | GPV_R4, H1b, GPV_K-n, H1b-K3n (B3MN definitions, normals cache) |
| **E1** | mean of H1b-K3n[LM0] and GPV_K-n[Z-LM0] counts |
| **E2** | mean of GPV_K-n[LM0] and GPV_K-n[Z-LM0] |
| **E3** | mean of H1b-K3n[LM0] and H1b-K3n[Z-LM0] |
| Targets | T1, T2′, T3′ |

## Run order
CPU → 1.0–1.3 → 2.1 → 3.1 → 4.1.

# 1. Setup

In [1]:
# 1.0 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ============================================================
# Step 1.1 — Paths, repo modules, train pool, frozen-system counts
# ============================================================

import importlib, json, subprocess, sys, time
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

PROJECT_ROOT = Path("/content/drive/MyDrive/MSc_Project_ZeroShot_Counting")
RESULTS_ROOT = PROJECT_ROOT / "results"
FSC147_RAW = PROJECT_ROOT / "datasets" / "FSC147" / "raw"
FSC147_IMAGE_DIR = FSC147_RAW / "images_384_VarV2"
TD_ROOT = PROJECT_ROOT / "cache" / "FSC147_train_dev400"
B3S_DIR = RESULTS_ROOT / "B3S_Semantic_Improvement"
OUT_DIR = RESULTS_ROOT / "B3MX_Mix_and_Ensemble"
OUT_DIR.mkdir(parents=True, exist_ok=True)

REPO_URL = "https://github.com/paulmcchan/Surrey-MScProject-Zero-Shot-Object-Counting.git"
REPO_DIR = Path("/content/Surrey-MScProject-Zero-Shot-Object-Counting")
!rm -rf "{REPO_DIR}"
!git clone -q "{REPO_URL}" "{REPO_DIR}"
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
REPO_COMMIT = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()

import src.pipeline.io_cache as io_cache
import src.pipeline.sg as sg
import src.pipeline.gpv as gpv
import src.pipeline.hybrid as hybrid
import src.pipeline.b0 as b0
for m in (io_cache, sg, gpv, hybrid, b0):
    importlib.reload(m)

STRATA = ["S1_0_20", "S2_20_40", "S3_40_60", "S4_60_80", "S5_80_95", "S6_95_100"]
W = {"S1_0_20": .20, "S2_20_40": .20, "S3_40_60": .20, "S4_60_80": .20, "S5_80_95": .15, "S6_95_100": .05}
TD_DIRS = {"b0_masks": TD_ROOT / "b0_masks", "a3c_masks": TD_ROOT / "a3c_masks",
           "depth": TD_ROOT / "depth", "m1_split_labels": TD_ROOT / "m1_split_labels"}
SG_CFG = sg.SGConfig.from_frozen_json(RESULTS_ROOT / "B3R_SG_Rebuild_and_Fixes" / "SG_v1.1_frozen_config.json")

td_df = pd.read_csv(TD_ROOT / "FSC147_train_dev400.csv")
td_res = pd.read_csv(B3S_DIR / "B3S_step1_per_image.csv")
assert len(td_df) == 400 and len(td_res) == 400
print("Repo commit:", REPO_COMMIT, "| output:", OUT_DIR)

Repo commit: d70bb1d6d631c12156ba7bf1881d22f5b2eb6100 | output: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B3MX_Mix_and_Ensemble


In [3]:
import torch

# ============================================================
# Step 1.2 — Shared helpers: guided masks, B0 meta filtering, systems, targets
# ============================================================

def guided_masks_for_prompts(predictor, image_np, prompts_xy):
    h, w = image_np.shape[:2]
    masks, scores = [], []
    if len(prompts_xy):
        with torch.inference_mode():
            predictor.set_image(image_np)
            for p in prompts_xy:
                m, s, _ = predictor.predict(point_coords=np.asarray([p], dtype=np.float32),
                                            point_labels=np.asarray([1], dtype=np.int32),
                                            multimask_output=False)
                masks.append(np.asarray(m[0], dtype=bool)); scores.append(float(np.asarray(s)[0]))
    return (np.asarray(masks) if masks else np.zeros((0, h, w), bool)), np.asarray(scores, float)

def filter_meta(meta, keep):
    return {k: np.asarray(v)[keep] for k, v in meta.items()}

def systems_from_masks(b0_masks, b0_iou, g_masks, g_scores, a3c, depth, san_target):
    keep = gpv.nms_masks(g_masks, g_scores, 0.5) if g_masks.shape[0] else np.zeros(0, int)
    r4_m, _, _ = gpv.build_r4(g_masks[keep], np.asarray(g_scores)[keep], b0_masks, b0_iou, 0.5)
    d1 = gpv.d1_peaks(san_target)
    h1b = float(hybrid.h1b_count(r4_m, d1, a3c, depth, SG_CFG)[0])
    return {"B0": int(b0_masks.shape[0]), "GPV_R4": int(r4_m.shape[0]), "H1b": h1b}

def evaluate(frame, systems, label):
    gt = frame["gt_count"].to_numpy(float); st = frame["density_stratum"].to_numpy()
    AE = {s: np.abs(frame[s].to_numpy(float) - gt) for s in systems}
    IDX = {}
    for j, k in enumerate(STRATA):
        pos = np.flatnonzero(st == k)
        IDX[k] = pos[np.random.default_rng(1000 + j).integers(0, len(pos), size=(10_000, len(pos)))]
    def sci(a, b, k):
        d = AE[a][IDX[k]].mean(1) - AE[b][IDX[k]].mean(1)
        return AE[a][st == k].mean() - AE[b][st == k].mean(), np.percentile(d, 2.5), np.percentile(d, 97.5)
    def rci(a, b):
        d = sum(w * (AE[a][IDX[k]].mean(1) - AE[b][IDX[k]].mean(1)) for k, w in W.items())
        o = sum(w * (AE[a][st == k].mean() - AE[b][st == k].mean()) for k, w in W.items())
        return o, np.percentile(d, 2.5), np.percentile(d, 97.5)
    res = pd.DataFrame([{"system": s,
                         "MAE_reweighted": sum(w * AE[s][st == k].mean() for k, w in W.items()),
                         "mean_signed": float((frame[s] - frame["gt_count"]).mean()),
                         **{f"MAE_{k[:2]}": AE[s][st == k].mean() for k in STRATA}} for s in systems])
    rows = []
    for s in [x for x in systems if x not in ("B0_v1", "GPV_R4_v1", "H1b_v1")]:
        rec = {"system": s}; t1 = []; t2 = []
        for k in ["S1_0_20", "S2_20_40"]:
            d, lo, hi = sci(s, "B0_v1", k); rec[f"{k[:2]} vs B0"] = f"{d:+.2f} [{lo:+.2f}, {hi:+.2f}]"; t1.append(lo <= 0)
        for k in ["S5_80_95", "S6_95_100"]:
            d, lo, hi = sci(s, "H1b_v1", k); rec[f"{k[:2]} vs H1b"] = f"{d:+.2f} [{lo:+.2f}, {hi:+.2f}]"; t2.append(lo <= 0)
        d, lo, hi = rci(s, "H1b_v1"); rec["overall vs H1b"] = f"{d:+.2f} [{lo:+.2f}, {hi:+.2f}]"
        rec.update({"T1": all(t1), "T2'": all(t2), "T3'": hi < 0, "ALL": all(t1) and all(t2) and hi < 0})
        rows.append(rec)
    print(f"{label} — metrics:"); display(res.round(2))
    print("Targets (difference [95% CI]; negative = system better):"); display(pd.DataFrame(rows))
    return res, pd.DataFrame(rows)
print("helpers ready")

helpers ready


In [4]:
# ============================================================
# Step 1.3 — Normals edge map and K3n counting (B3MN definitions)
# ============================================================

from scipy import ndimage
import src.pipeline.multiplicity as K
importlib.reload(K)
N_ROOT = TD_ROOT / "normals"

def normal_edge_map(nrm):
    n = nrm.astype(np.float32)
    n = n / np.maximum(np.linalg.norm(n, axis=2, keepdims=True), 1e-6)
    edge = np.zeros(n.shape[:2], np.float32)
    for dy, dx in [(0, 1), (1, 0), (0, -1), (-1, 0)]:
        edge = np.maximum(edge, 1.0 - (n * np.roll(n, shift=(dy, dx), axis=(0, 1))).sum(axis=2))
    edge[0, :] = edge[1, :]; edge[-1, :] = edge[-2, :]; edge[:, 0] = edge[:, 1]; edge[:, -1] = edge[:, -2]
    return ndimage.gaussian_filter(edge, 1.0)

def k3n_segments(region, edge, min_seg, h_frac=0.1):
    e = edge[region]
    h = h_frac * float(np.percentile(e, 95) - np.percentile(e, 5))
    return 1 if h <= 0 else K._marker_watershed(edge, region, -edge, h, min_seg)

def gpv_k_n_count(r4_m, r4_s, d1, edge):
    masks = np.asarray(r4_m, bool)
    if masks.shape[0] == 0:
        return 0
    flagged = gpv.merged_mask_flags(masks, gpv.points_per_mask(masks, d1), "A")
    a_ref = K.reference_area(masks, flagged)
    n_nonflag = int((~flagged).sum())
    regions = [rg for _, rg in K.build_regions(masks, r4_s, flagged) if rg.sum() > 0 and rg.sum() >= 0.5 * a_ref]
    if not (regions and sum(K.k1_peaks(rg, d1) for rg in regions) > n_nonflag):
        return int(masks.shape[0])
    return n_nonflag + sum(max(1, k3n_segments(rg, edge, 0.25 * a_ref)) for rg in regions)

def h1b_k3n_count(r4_m, d1, a3c, depth, edge):
    base, info = hybrid.h1b_count(r4_m, d1, a3c, depth, SG_CFG)
    if not info["stage2_on"] or r4_m.shape[0] == 0:
        return base
    merged = gpv.merged_mask_flags(r4_m, gpv.points_per_mask(r4_m, d1), "A")
    if not merged.any():
        return base
    anchor = r4_m[~merged]
    areas = anchor.reshape(anchor.shape[0], -1).sum(1) if anchor.shape[0] else np.zeros(0)
    a_ref = float(np.median(areas)) if areas.size else float(np.median(r4_m[merged].reshape(merged.sum(), -1).sum(1)))
    taken = anchor.any(axis=0) if anchor.shape[0] else np.zeros(r4_m.shape[1:], bool)
    total = int(anchor.shape[0])
    for i in np.flatnonzero(merged):
        rg = r4_m[i] & ~taken
        taken |= rg
        if rg.sum() > 0 and rg.sum() >= 0.5 * a_ref:
            total += max(1, k3n_segments(rg, edge, 0.25 * a_ref))
    return float(total)

def all_counts(masks, scores, d1, a3c, depth, edge):
    return {"GPV_R4": int(masks.shape[0]),
            "H1b": float(hybrid.h1b_count(masks, d1, a3c, depth, SG_CFG)[0]),
            "GPV_K-n": gpv_k_n_count(masks, scores, d1, edge),
            "H1b-K3n": h1b_k3n_count(masks, d1, a3c, depth, edge)}
print("K3n counting ready")

K3n counting ready


# 2. Mask-source mixing

In [5]:
# ============================================================
# Step 2.1 — MixA and MixB on train_dev400 (CPU, cached masks)
# ============================================================

LM_ROOT = TD_ROOT / "sam2_large_masked_M0"
ZLM_ROOT = TD_ROOT / "zoomed" / "LM0"
assert len(list((ZLM_ROOT / "_done").glob("*.json"))) == 400 and len(list((LM_ROOT / "_done").glob("*.json"))) == 400

def r4_from(b0_dir, guided_dir, di, stem):
    bm, bmeta = io_cache.load_b0_masks(b0_dir / "b0" / f"{int(di):05d}_{stem}.npz")
    g = io_cache.load_guided_masks(guided_dir / "guided" / f"{int(di):04d}_{stem}.npz")
    kg = gpv.nms_masks(g["masks"], g["scores"], 0.5) if g["masks"].shape[0] else np.zeros(0, int)
    return gpv.build_r4(g["masks"][kg], np.asarray(g["scores"])[kg], bm, bmeta["predicted_iou"], 0.5)[:2]

rows = []
t0 = time.time()
for r in tqdm(td_df.itertuples(index=False), total=len(td_df), desc="Mixes"):
    di, stem = r.dataset_index, io_cache.image_stem(r.image_id)
    p = io_cache.cache_paths(di, r.image_id, TD_DIRS)
    a3c, _ = io_cache.load_a3c_mask(p["a3c_masks"])
    dep = io_cache.load_depth(p["depth"])
    san = io_cache.load_san_scores(TD_ROOT / "san" / io_cache.san_scores_filename(r.image_id))
    d1 = gpv.d1_peaks(san["target_score"])
    with np.load(N_ROOT / f"{stem}.npz") as v:
        edge = normal_edge_map(v["normals"])
    row = {"dataset_index": int(di)}
    for name, bdir, gdir in [("MixA", ZLM_ROOT, LM_ROOT), ("MixB", LM_ROOT, ZLM_ROOT)]:
        m, s = r4_from(bdir, gdir, di, stem)
        row.update({f"{k}[{name}]": v for k, v in all_counts(m, s, d1, a3c, dep, edge).items()})
    rows.append(row)
mix = pd.DataFrame(rows)
print(f"Runtime: {(time.time() - t0) / 60:.1f} min")

Mixes:   0%|          | 0/400 [00:00<?, ?it/s]

Runtime: 66.9 min


# 3. Count ensembles

In [6]:
# ============================================================
# Step 3.1 — Ensembles E1–E3 from saved per-image counts
# ============================================================

mn = pd.read_csv(RESULTS_ROOT / "B3MN_Marigold_Normals" / "section4_bases" / "B3MN_s4_per_image.csv")
need = ["H1b-K3n[LM0]", "GPV_K-n[LM0]", "GPV_K-n[Z-LM0]", "H1b-K3n[Z-LM0]", "H1b[Z-LM0]"]
assert all(c in mn.columns for c in need), [c for c in need if c not in mn.columns]
ens = mn[["dataset_index"] + need].copy()
ens["E1"] = 0.5 * (ens["H1b-K3n[LM0]"] + ens["GPV_K-n[Z-LM0]"])
ens["E2"] = 0.5 * (ens["GPV_K-n[LM0]"] + ens["GPV_K-n[Z-LM0]"])
ens["E3"] = 0.5 * (ens["H1b-K3n[LM0]"] + ens["H1b-K3n[Z-LM0]"])
print("Ensembles built for", len(ens), "images")

Ensembles built for 400 images


# 4. Results and targets

In [7]:
# ============================================================
# Step 4.1 — Metrics and targets; error-sign check for the ensembles
# ============================================================

frame = (mix.merge(ens, on="dataset_index", validate="one_to_one")
         .merge(td_res[["dataset_index", "gt_count", "density_stratum", "B0", "GPV_R4_v1", "H1b_v1"]]
                .rename(columns={"B0": "B0_v1"}), on="dataset_index", validate="one_to_one"))

print("Share of images where the two ensemble members err in OPPOSITE directions (by stratum):")
sign = {}
for e, (a, b) in {"E1": ("H1b-K3n[LM0]", "GPV_K-n[Z-LM0]"), "E2": ("GPV_K-n[LM0]", "GPV_K-n[Z-LM0]"),
                  "E3": ("H1b-K3n[LM0]", "H1b-K3n[Z-LM0]")}.items():
    ea, eb = frame[a] - frame["gt_count"], frame[b] - frame["gt_count"]
    sign[e] = ((ea * eb) < 0).groupby(frame["density_stratum"]).mean()
display(pd.DataFrame(sign).reindex(STRATA).round(2))

SYS = ["B0_v1", "GPV_R4_v1", "H1b_v1", "H1b-K3n[LM0]", "GPV_K-n[LM0]", "GPV_K-n[Z-LM0]", "H1b-K3n[Z-LM0]"] + \
      [f"{k}[{m}]" for m in ["MixA", "MixB"] for k in ["GPV_R4", "H1b", "GPV_K-n", "H1b-K3n"]] + ["E1", "E2", "E3"]
res, targets = evaluate(frame, SYS, "Mixing and ensembling (train_dev400)")

frame.to_csv(OUT_DIR / "B3MX_per_image.csv", index=False)
res.to_csv(OUT_DIR / "B3MX_metrics.csv", index=False)
targets.to_csv(OUT_DIR / "B3MX_targets.csv", index=False)
with open(OUT_DIR / "B3MX_metadata.json", "w") as f:
    json.dump({"notebook": "B3MX_Mix_and_Ensemble",
               "MixA": "B0 masks Z-LM0 + guided masks LM0", "MixB": "B0 masks LM0 + guided masks Z-LM0 (diagnostic)",
               "ensembles": {"E1": "mean(H1b-K3n[LM0], GPV_K-n[Z-LM0])", "E2": "mean(GPV_K-n[LM0], GPV_K-n[Z-LM0])",
                             "E3": "mean(H1b-K3n[LM0], H1b-K3n[Z-LM0])"},
               "date_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
               "repo_commit": REPO_COMMIT, "heldout_accessed": False, "test_set_accessed": False}, f, indent=2)
print("Saved to:", OUT_DIR)

Share of images where the two ensemble members err in OPPOSITE directions (by stratum):


,E1,E2,E3
density_stratum,,,
S1_0_20,0.04,0.02,0.02
S2_20_40,0.08,0.06,0.08
S3_40_60,0.12,0.10,0.08
S4_60_80,0.20,0.22,0.17
S5_80_95,0.30,0.30,0.22
S6_95_100,0.10,0.08,0.15


Mixing and ensembling (train_dev400) — metrics:


,system,MAE_reweighted,mean_signed,MAE_S1,MAE_S2,MAE_S3,MAE_S4,MAE_S5,MAE_S6
0,B0_v1,35.67,-62.67,5.56,7.74,13.30,22.93,59.03,338.08
1,GPV_R4_v1,29.94,-46.78,8.62,7.70,12.45,16.13,40.52,297.63
2,H1b_v1,28.65,-41.28,10.46,9.49,12.35,14.20,37.33,274.98
3,H1b-K3n[LM0],28.24,-44.33,6.92,6.24,11.62,15.55,38.97,286.55
4,GPV_K-n[LM0],27.71,-41.95,8.31,5.62,10.73,15.57,39.80,273.78
5,GPV_K-n[Z-LM0],27.02,-30.56,13.98,9.43,15.35,16.43,30.90,227.00
6,H1b-K3n[Z-LM0],27.25,-28.88,10.66,8.90,17.23,16.67,30.67,239.22
7,GPV_R4[MixA],27.02,-39.64,7.98,7.29,12.12,15.67,37.47,255.92
8,H1b[MixA],26.62,-35.75,9.66,8.31,12.85,16.13,32.92,245.75
9,GPV_K-n[MixA],26.11,-32.99,13.02,8.22,12.18,16.02,29.88,234.73


Targets (difference [95% CI]; negative = system better):


,system,S1 vs B0,S2 vs B0,S5 vs H1b,S6 vs H1b,overall vs H1b,T1,T2',T3',ALL
0,H1b-K3n[LM0],"+1.36 [-0.39, +3.10]","-1.50 [-3.26, +0.29]","+1.63 [-5.00, +8.97]","+11.57 [-9.08, +31.38]","-0.41 [-2.10, +1.39]",True,True,False,False
1,GPV_K-n[LM0],"+2.75 [+0.25, +5.49]","-2.11 [-3.86, -0.24]","+2.47 [-6.15, +12.07]","-1.20 [-28.62, +23.70]","-0.94 [-3.05, +1.25]",False,True,False,False
2,GPV_K-n[Z-LM0],"+8.41 [+4.65, +12.46]","+1.69 [-0.83, +4.45]","-6.43 [-14.73, +2.92]","-47.98 [-76.67, -21.10]","-1.63 [-3.91, +0.73]",False,True,False,False
3,H1b-K3n[Z-LM0],"+5.10 [+2.62, +7.78]","+1.16 [-1.26, +3.78]","-6.67 [-13.27, +0.98]","-35.77 [-56.47, -14.38]","-1.40 [-3.25, +0.58]",False,True,False,False
4,GPV_R4[MixA],"+2.41 [+0.54, +4.40]","-0.45 [-2.44, +1.64]","+0.13 [-7.00, +8.73]","-19.07 [-36.48, -0.63]","-1.62 [-3.31, +0.15]",False,True,False,False
5,H1b[MixA],"+4.10 [+1.87, +6.47]","+0.58 [-1.72, +3.00]","-4.42 [-10.43, +2.83]","-29.23 [-47.95, -9.48]","-2.03 [-3.66, -0.31]",False,True,True,False
6,GPV_K-n[MixA],"+7.46 [+3.78, +11.47]","+0.49 [-1.89, +3.08]","-7.45 [-16.20, +2.42]","-40.25 [-65.99, -15.67]","-2.54 [-4.76, -0.26]",False,True,True,False
7,H1b-K3n[MixA],"+4.21 [+1.92, +6.63]","+0.65 [-1.66, +3.07]","-4.55 [-10.83, +2.82]","-32.52 [-53.28, -10.93]","-1.66 [-3.53, +0.32]",False,True,False,False
8,GPV_R4[MixB],"+1.38 [-0.35, +3.13]","-1.61 [-3.31, +0.12]","+7.87 [+1.43, +16.02]","+29.78 [+17.35, +43.55]","+1.72 [+0.27, +3.30]",True,False,False,False
9,H1b[MixB],"+2.14 [+0.22, +4.14]","-0.85 [-2.71, +1.11]","+4.00 [-1.02, +10.47]","+18.00 [+5.83, +31.80]","+0.65 [-0.64, +2.05]",False,False,False,False


Saved to: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B3MX_Mix_and_Ensemble
